# Sesión 3 · Limpieza de Datos — Proyecto MIMETIC

**Objetivo:** transformar el dataset crudo `dailyActivity_mergedMIMETIC(version 1).xlsx` en una versión limpia (`df_clean`) y reproducible, siguiendo la metodología CRISP-DM (fase de *Preparación de los Datos*).

**Principio rector:** nuestro DataFrame original nunca se modifica. Cada corrección se aplica sobre una copia de trabajo (`df_clean`), y cada paso se documenta con QUÉ hace y POR QUÉ existe.

## 0. Imports

Importamos las librerías necesarias: `pandas` para manipular tablas, `requests` para consumir la API y `json` para interpretar su respuesta.

In [1]:
import json
import pandas as pd
import requests

## 1. Carga de la Fuente 1 — Dataset de sensores (Excel)

`pd.read_excel` lee la hoja y crea un DataFrame llamado `df_mimetic`. Esta es la **tabla cruda**: todavía no se limpia nada, solo se define la referencia sobre la que trabajaremos.

In [2]:
df_mimetic = pd.read_excel("dailyActivity_mergedMIMETIC(version 1).xlsx")

In [3]:
print("Dimensiones (Filas, Columnas):", df_mimetic.shape)
print("\nTipos de datos detectados:\n", df_mimetic.dtypes)
print("\nPrimeros 3 registros:\n", df_mimetic.head(3))

Dimensiones (Filas, Columnas): (940, 17)

Tipos de datos detectados:
 Id                            int64
ActivityDate                 object
TotalSteps                    int64
TotalDistance               float64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance     float64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
Nivel_Pasos                     str
Nivel_Sedentarismo              str
Unnamed: 15                 float64
Unnamed: 16                     str
dtype: object

Primeros 3 registros:
            Id         ActivityDate  TotalSteps  TotalDistance  \
0  1503960366  2016-12-04 00:00:00       13162   8.500000e+00   
1  1503960366            4/13/2016       10735   6.970000e+14   
2  1503960366            4/14/2016       10460   6.740000e+14   

   VeryActiveDistan

## 2. Carga de la Fuente 2 — API REST OpenWeatherMap (JSON)

Esta fuente es semiestructurada: la API responde en formato JSON y nosotras la convertimos a tabla con `pd.DataFrame(clima_datos)`. Solo se usa como contexto complementario (condiciones climáticas por ciudad).

In [4]:
api_key = "d7ec6c327fa27f4f6b8441a33d7a2075"
ciudades = ["Bogota", "Medellin", "Cali", "Ibague"]
clima_datos = []

for ciudad in ciudades:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={ciudad}&appid={api_key}&units=metric&lang=es"
    try:
        res = requests.get(url)
        if res.status_code == 200:
            data = res.json()
            clima_datos.append({
                "Ciudad": data["name"],
                "Temperatura_C": data["main"]["temp"],
                "Humedad_Pct": data["main"]["humidity"],
                "Estado_Clima": data["weather"][0]["description"],
                "Velocidad_Viento": data["wind"]["speed"]
            })
        else:
            print(f"Error {res.status_code} al consultar {ciudad}")
    except Exception as e:
        print(f"Error de conexión al consultar {ciudad}: {e}")

df_api_clima = pd.DataFrame(clima_datos)
print("\n=== FUENTE 2: DATOS CLIMÁTICOS (API REST / JSON) ===")
print(df_api_clima)


=== FUENTE 2: DATOS CLIMÁTICOS (API REST / JSON) ===
             Ciudad  Temperatura_C  Humedad_Pct   Estado_Clima  \
0            Bogota          17.75           59  lluvia ligera   
1          Medellín          24.49           59          nubes   
2  Santiago de Cali          30.00           61  lluvia ligera   
3            Ibagué          27.95           39  lluvia ligera   

   Velocidad_Viento  
0              9.26  
1              2.68  
2              2.06  
3              3.60  


---
# PIPELINE DE LIMPIEZA

A partir de aquí trabajamos **siempre sobre `df_clean`** (una copia), nunca sobre `df_mimetic`. El orden de los pasos no es caprichoso: duplicados e inconsistencias se resuelven **antes** de imputar o recortar, porque contaminan los cálculos posteriores (medianas, cuantiles, modas).

## PASO 0 · Diagnóstico de calidad

Diagnosticar es gratis y revela al paciente antes de operar. Tres preguntas:
* `isna().sum()` — ¿cuánto y dónde falta?
* `duplicated().sum()` — ¿hay filas repetidas exactas?
* `describe()` / inspección manual — ¿hay valores fuera de rango?

**Por qué `df_mimetic.isna().sum()[lambda s: s > 0]`:** `isna().sum()` devuelve una Serie con el conteo de nulos por columna; el filtro `[lambda s: s > 0]` se queda solo con las columnas que efectivamente tienen nulos (evita ruido con las que tienen 0).

In [5]:
print("=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===")
print(f"Dimensiones: {df_mimetic.shape}")
print(f"Nulos explícitos (isna): {df_mimetic.isna().sum().sum()}")
print(f"Duplicados exactos: {df_mimetic.duplicated().sum()}")
print("\nNulos por columna:")
print(df_mimetic.isna().sum()[lambda s: s > 0])
print("\nColumnas fantasmas (artefactos de Excel):", [c for c in df_mimetic.columns if "Unnamed" in c])
print("\nDías de 'no uso' del tracker (0 pasos y 1440 min sedentarios):",
      ((df_mimetic["TotalSteps"] == 0) & (df_mimetic["SedentaryMinutes"] == 1440)).sum())

=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===
Dimensiones: (940, 17)
Nulos explícitos (isna): 1879
Duplicados exactos: 0

Nulos por columna:
Unnamed: 15    940
Unnamed: 16    939
dtype: int64

Columnas fantasmas (artefactos de Excel): ['Unnamed: 15', 'Unnamed: 16']

Días de 'no uso' del tracker (0 pasos y 1440 min sedentarios): 72


### Hallazgos del diagnóstico
1. **2 columnas fantasma** (`Unnamed: 15`, `Unnamed: 16`): el Excel guardó contenido fuera del rango de la tabla y pandas las creó automáticamente. Son las únicas con nulos (1879 en total).
2. **0 duplicados exactos**: buena noticia, no hay que deduplicar.
3. **72 días de 'no uso'**: registros donde el usuario no portó el dispositivo (0 pasos y 24h sentado). Son **MNAR**: la ausencia física del día explica el dato ausente. Imputarlos sería inventar actividad que no ocurrió → se eliminan.

## PASO 1 · Eliminación de columnas fantasma

**QUÉ:** se detectan las columnas cuyo nombre contiene `"Unnamed"` y se eliminan con `drop(columns=...)`.
**POR QUÉ:** son basura creada por Excel al escribir la hoja; no aportan información y son la fuente de los 1879 nulos.

**Por qué `.copy()`:** `drop()` devuelve una vista en algunos casos. La copia evita el *SettingWithCopyWarning* y garantiza que `df_clean` sea un DataFrame independiente que sí podemos modificar.

In [6]:
cols_fantasma = [c for c in df_mimetic.columns if "Unnamed" in c]
df_clean = df_mimetic.drop(columns=cols_fantasma).copy()
print(f"[PASO 1] Columnas fantasma eliminadas: {cols_fantasma}")
print(f"Nuevas dimensiones: {df_clean.shape}")

[PASO 1] Columnas fantasma eliminadas: ['Unnamed: 15', 'Unnamed: 16']
Nuevas dimensiones: (940, 15)


## PASO 2 · Normalización de fechas

**QUÉ:** `pd.to_datetime(..., errors="coerce")` convierte `ActivityDate` de texto a tipo fecha.
**POR QUÉ:** el columnas mezclan formatos (`2016-12-04`, `4/13/2016`). Como fecha real, pandas puede ordenar, extraer día/mes/año y calcular series de tiempo.

**Por qué `errors="coerce"`:** si hay una fecha ilegible la convierte en `NaT` (nulo de fechas) en lugar de reventar el programa; los nulos se reportan y deciden aparte.

In [7]:
df_clean["ActivityDate"] = pd.to_datetime(df_clean["ActivityDate"], errors="coerce")
fechas_invalidas = df_clean["ActivityDate"].isna().sum()
print(f"[PASO 2] 'ActivityDate' convertida a datetime. Fechas inválidas (NaT): {fechas_invalidas}")

[PASO 2] 'ActivityDate' convertida a datetime. Fechas inválidas (NaT): 0


## PASO 3 · Eliminación de días de 'no uso' del tracker (MNAR)

**QUÉ:** se identifican las filas donde `TotalSteps == 0` y `SedentaryMinutes == 1440` (24 h sentado, es decir, sin portar el dispositivo) y se eliminan con `drop(index=...)`.
**POR QUÉ:** son **MNAR** (el dato falta porque ese día el evento simplemente no existió). Imputar pasos a un día sin dispositivo distorsionaría el promedio de actividad y de sedentarismo real de la persona.

**Por qué `reset_index(drop=True)`:** al borrar filas los índices quedan con huecos; reindexarlos de 0 a N-1 mantiene el dataset consistente y evita problemas al filtrar después.

In [8]:
indices_no_uso = df_clean[
    (df_clean["TotalSteps"] == 0) & (df_clean["SedentaryMinutes"] == 1440)
].index

df_clean = df_clean.drop(index=indices_no_uso).reset_index(drop=True)
print(f"[PASO 3] Registros de 'No Uso' eliminados: {len(indices_no_uso)}")
print(f"Filas restantes: {len(df_clean)}")

[PASO 3] Registros de 'No Uso' eliminados: 72
Filas restantes: 868


## PASO 4 · Función de tratamiento de outliers con la Regla IQR

**QUÉ hace la función (método de Tukey):**
1. Calcula los cuartiles `Q1` (percentil 25) y `Q3` (percentil 75).
2. Deriva el rango intercuartílico `IQR = Q3 - Q1` (la caja central donde vive el 50% de los datos).
3. Define los límites aceptables: `lim_inf = Q1 - 1.5·IQR` y `lim_sup = Q3 + 1.5·IQR`.
4. Cuenta cuántos valores quedan fuera de esos límites (candidatos a outlier).
5. Aplica **winsorización** con `clip()`: el valor se *recorta* al límite, no se borra la fila.

**POR QUÉ winsorizar y no eliminar:** conservamos al cliente/día y su contexto, pero acallamos el 'escándalo numérico'. Eliminar filas solo cuando el outlier es imposible o son pocas y el error está documentado.

**POR QUÉ `lower=max(0, lim_inf)`:** variables como pasos o calorías no pueden ser negativas; si el límite inferior sale negativo se aplana a 0.

> **Nota:** la función debe estar **completa en una sola celda**. Si se parte, las variables internas (`df_in`, `columna`, `lim_inf`...) no existen fuera de la celda y Python lanza `NameError`. Además, para modificar el DataFrame dentro de la función hay que **asignarlo de vuelta** (`df_clean = tratar_outliers_iqr(df_clean, col)`), porque pandas no muta automáticamente.

In [9]:
def tratar_outliers_iqr(df_in, columna, factor=1.5):
    """Recorta (winsoriza) los valores extremos de una columna numérica
    usando la regla IQR. Devuelve el mismo DataFrame modificado."""
    # 1-2. Cuartiles y rango intercuartílico
    q1 = df_in[columna].quantile(0.25)
    q3 = df_in[columna].quantile(0.75)
    iqr = q3 - q1

    # 3. Límites aceptables (factor 1.5 por defecto, regla de Tukey)
    lim_inf = q1 - (factor * iqr)
    lim_sup = q3 + (factor * iqr)
    lim_inf_real = max(0, lim_inf)  # las métricas físicas no son negativas

    # 4. Conteo de valores fuera de límites
    fuera = df_in.loc[(df_in[columna] < lim_inf) | (df_in[columna] > lim_sup)].shape[0]

    # 5. Winsorización: recortar al límite permitido
    df_in[columna] = df_in[columna].clip(lower=lim_inf_real, upper=lim_sup)

    print(f"  -> '{columna}': IQR={iqr:.2f} | límites=[{lim_inf_real:.2f}, {lim_sup:.2f}] | outliers recortados={fuera}")
    return df_in

## PASO 5 · Aplicar la regla IQR a las variables numéricas

**QUÉ:** se recorre una lista de columnas y se les aplica la función anterior.
**POR QUÉ esas tres:** `TotalSteps`, `Calories` y `VeryActiveMinutes` son métricas de esfuerzo físico donde un valor disparado (un error de registro o un día de maratón imposible) torcería la media y los modelos.

> **Ojo con `TotalDistance`:** su magnitud llega a `6.97e14` (ciento de billones), un **error sistemático de escala** (probablemente científicación mal interpretada por Excel), no outliers aislados. En un proyecto real este columna se revisaría con el negocio o se reconstruiría desde las fuentes; la regla IQR sola no alcanza. Lo dejamos documentado aquí como hallazgo.

In [10]:
print("[PASO 5] Aplicando la regla IQR (winsorización):")
for col in ["TotalSteps", "Calories", "VeryActiveMinutes"]:
    df_clean = tratar_outliers_iqr(df_clean, col)

[PASO 5] Aplicando la regla IQR (winsorización):
  -> 'TotalSteps': IQR=6230.25 | límites=[0.00, 20400.38] | outliers recortados=15
  -> 'Calories': IQR=976.75 | límites=[388.62, 4295.62] | outliers recortados=11
  -> 'VeryActiveMinutes': IQR=35.00 | límites=[0.00, 87.50] | outliers recortados=56


## PASO 6 · Consistencia en variables categóricas

**QUÉ:** `str.strip()` elimina espacios en blanco al inicio/fin de cada etiqueta y luego mostramos los valores únicos con `unique()`.
**POR QUÉ:** "Alta Actividad " (con espacio) y "Alta Actividad" serían tratadas como categorías distintas por SQL y por el one-hot encoding de la próxima sesión. Normalizar antes de modelar previene bugs silenciosos.

In [11]:
for col in ["Nivel_Pasos", "Nivel_Sedentarismo"]:
    df_clean[col] = df_clean[col].str.strip()
    print(f"[PASO 6] '{col}' -> valores únicos: {sorted(df_clean[col].unique())}")

[PASO 6] 'Nivel_Pasos' -> valores únicos: ['Alta Actividad', 'Baja Actividad']
[PASO 6] 'Nivel_Sedentarismo' -> valores únicos: ['Alto Sedentarismo', 'Bajo Sedentarismo']


## PASO 7 · Validación final y exportación

**QUÉ:** verificamos que la limpieza cumplió sus metas (0 nulos, 0 duplicados, dimensiones correctas) y revisamos las estadísticas resultantes.
**POR QUÉ:** la validación contra criterios explícitos es lo que hace el pipeline **reproducible**: si al volver a ejecutarlo estas cifras cambian, el pipeline cambió.

**Por qué `describe().round(1)`:** `describe()` resume media, cuartiles, min/max; `.round(1)` controla los decimales para que el reporte sea legible. Los máximos ya no deberían ser absurdos tras la winsorización.

In [12]:
print("=== VALIDACIÓN FINAL (df_clean) ===")
print(f"Dimensiones: {df_clean.shape}")
print(f"Nulos totales: {df_clean.isna().sum().sum()}")
print(f"Duplicados exactos: {df_clean.duplicated().sum()}")
print("\nEstadísticas finales de las métricas de actividad:")
print(df_clean[["TotalSteps", "Calories", "VeryActiveMinutes"]].describe().round(1))

=== VALIDACIÓN FINAL (df_clean) ===
Dimensiones: (868, 15)
Nulos totales: 0
Duplicados exactos: 0

Estadísticas finales de las métricas de actividad:
       TotalSteps  Calories  VeryActiveMinutes
count       868.0     868.0              868.0
mean       8211.9    2352.5               21.0
std        4565.6     702.5               27.6
min           0.0     388.6                0.0
25%        4824.8    1853.8                0.0
50%        7969.0    2219.0                7.0
75%       11055.0    2830.5               35.0
max       20400.4    4295.6               87.5


In [13]:
# Exportación del dataset limpio (la versión original nunca se sobrescribe)
df_clean.to_csv("MIMETIC_limpio.csv", index=False)
print(f"Dataset limpio exportado: MIMETIC_limpio.csv ({df_clean.shape[0]} filas)")

Dataset limpio exportado: MIMETIC_limpio.csv (868 filas)


## Resumen de la bitácora de limpieza

| Paso | Qué se hizo | Método / Justificación | Cantidad total |
|---|---|---|---|
| 0 | Diagnóstico | `isna()`, `duplicated()`, inspección | 1879 nulos (solo columnas fantasma), 0 duplicados |
| 1 | Columnas fantasma `Unnamed` | `drop()` — artefactos de Excel, sin información | -2 columnas |
| 2 | `ActivityDate` a datetime | `to_datetime(errors="coerce")` — formatos mixtos | 0 inválidas |
| 3 | Días de no uso del tracker | `drop(index=...)` — MNAR, el dato no existió | -72 filas |
| 4-5 | Outliers IQR | Winsorización con `clip()` — regla de Tukey, conservar la fila | recorte en 3 columnas |
| 6 | Categóricas | `str.strip()` — evitar categorías duplicadas por espacios | 2 columnas |

**Hallazgo pendiente de revisión con el negocio:** `TotalDistance` y sus variantes muestran magnitudes del orden de `10^14` (error sistemático de escala). La winsorización no lo corrige porque no son outliers aislados; requiere validación de la fuente.